# ボラサーフェスの戦略ノート(JPY OISスワップション)

2026-10-28〜29に行った「PCA×ガンマ・ボラの傾向」の探索を、手元で動かして確かめるためのノートです。
データの可視化から始めて、見つかった5つの傾向がそれぞれ**どういう取引か**・**どう計算しているか**を順に追います。

| # | 取引 | 一言で |
|---|---|---|
| ① | 1Mのストラドル売り(デルタヘッジ) | 1MのIVは、その後1か月の実現ボラより高い |
| ② | ①のタイミング | 1M/1YのIV比が高い日ほど、①がよく儲かる |
| ③ | PC2売り | 1Mx7Y買い / 1Mx2Y+1Mx20Y売り(PC1のガンマ中立)+1Yx7Yでネットのベガを0に |
| ④ | 日銀会合の前後 | 会合の3日前に1Mx20Yを売り、3日後に買い戻す |
| ⑤ | 週末・IVの形 | 金曜に売って月曜に買い戻す / IVの年限方向の形の平均回帰 |

**単位**: P&Lはすべて「建てた時のベガ1単位あたり」で、ボラのbp(≒ 実現ボラ − IV)。ロング・ストラドルが +。
売りの戦略はその符号を反転して表示します。

**注意**
- 売買コストは既定で0です(下の `COST_BP` で片道のボラbpを入れると、控除後の数字も出ます)。
- 閾値の後付け・多重比較に注意(③の「比>1」は2025-04以降のデータで決め、それより前は本当の検証外)。
- 使うのは `data/historical.db` の `swaption_premiums`(Traditionの引け、2022-01〜)と `swaption_forwards`(年金の補正用)。
  **出力(数値・グラフ)にはLSEG・Traditionのデータが入るので、出力を残したままコミットしない**(検査で止まります)。

実行: `~/.venvs/pricing-sheet-win/Scripts/jupyter lab notebooks/vol_strategies.ipynb`
(必要なパッケージは `notebooks/requirements.txt`)。部品は同じフォルダの `vol_lab.py`。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in (ROOT, ROOT / "notebooks"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

import vol_lab
importlib.reload(vol_lab)
from vol_lab import *

# 色: 系列は固定の順番(青・橙・緑青…)、プラス/マイナスは青/赤、水準は青の単色
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
POS, NEG, GRAY = "#2a78d6", "#e34948", "#8a8984"
SEQ = LinearSegmentedColormap.from_list("seq", ["#eef4fc", "#2a78d6", "#0b2f5c"])
DIV = LinearSegmentedColormap.from_list("div", ["#e34948", "#f2f1ee", "#2a78d6"])
plt.rcParams.update({
    "font.family": ["Meiryo", "Yu Gothic", "sans-serif"],
    "axes.prop_cycle": plt.cycler(color=C), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6, "lines.linewidth": 1.5,
    "figure.figsize": (11, 4), "figure.dpi": 100, "axes.titlesize": 12, "legend.frameon": False,
})
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 200)

def bars(ax, s, title="", fmt="{:+.1f}"):
    # プラスは青・マイナスは赤の横棒、値のラベル付き
    s = s.dropna()
    ax.barh(range(len(s)), s.values, color=[POS if v >= 0 else NEG for v in s.values], height=0.6)
    ax.set_yticks(range(len(s)), s.index)
    ax.invert_yaxis()
    ax.axvline(0, color=GRAY, lw=0.8)
    for i, v in enumerate(s.values):
        ax.text(v, i, " " + fmt.format(v) + " ", va="center", ha="left" if v >= 0 else "right", fontsize=9)
    ax.set_title(title)
    ax.grid(axis="y", visible=False)

# ---- パラメータ(ここを変えて動かす) ----
DB = ROOT / "data" / "historical.db"
COST_BP = 0.0      # 売買コスト(片道、ミッドからのボラbp。ベガ1単位あたり)
TS = ["1Y", "2Y", "5Y", "7Y", "10Y", "20Y"]   # 表に出す原資産

## 1. データ

Traditionのストラドルのプレミアム(bp)とATMストライク(=CMのFwd)から、IVを逆算します。

- σ√T(bp)= プレミアム / (2 × 年金 × φ(0))。ATMのノーマル・ストラドルの価値 = 2σ√T·φ(0)·年金 だから。
- 年金は、フラットのFwd(年1回払い)で近似し、自前のカーブの正確な年金(2024-10〜)との比で満期x原資産ごとに補正。
- IV(営業日ベース)= σ√T / √(満期までの営業日数) × √252。
- 自前のIV(`vol_quotes`、自前のカーブで逆算)との差で、作り方を確かめます。

In [ ]:
quotes, info = load_quotes(DB)
S = Surface(quotes)
print(f"期間 {S.D[0].date()} 〜 {S.D[-1].date()}({S.nD}営業日)  原資産 {S.tenors}  満期 {GRID}")
d = info["vs_own"]
err = d.vol - d.v_own
print(f"自前のIVとの差(bp、{d.d.min().date()}〜): 平均 {err.mean():+.3f}  標準偏差 {err.std():.3f}  最大 {err.abs().max():.2f}")
print("年金の補正比(正確 / フラット近似)の分布:")
info["annuity_ratio"]["mean"].describe().to_frame("比").T

## 2. 可視化: サーフェスの形と時間の変化

### 2-1. ある日のIVサーフェス(満期 × 原資産)
`DATE` を変えて、形の変化を見てください。満期が短いほど・原資産が長いほどIVがどう並ぶかが、以降の取引の前提です。

In [ ]:
DATE = S.D[-1]          # 例: pd.Timestamp("2024-08-05")
DATE = S.D[S.D.searchsorted(pd.Timestamp(DATE))]
surf = S.surface_on(DATE)
fig, ax = plt.subplots(figsize=(10, 3.6))
im = ax.imshow(surf.values, cmap=SEQ, aspect="auto")
ax.set_xticks(range(len(surf.columns)), surf.columns)
ax.set_yticks(range(len(surf.index)), surf.index)
ax.set_xlabel("原資産"); ax.set_ylabel("満期"); ax.grid(False)
for i in range(surf.shape[0]):
    for j in range(surf.shape[1]):
        v = surf.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=9,
                    color="white" if (v - np.nanmin(surf.values)) / np.ptp(surf.values[np.isfinite(surf.values)]) > 0.45 else "#0b0b0b")
ax.set_title(f"IV(営業日ベース、bp/年) {DATE.date()}")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

### 2-2. IVの推移と、1M/1YのIV比
上: 1MのIV(実線)と1YのIV(点線)。下: 1M/1YのIV比(②の信号)。比が1を超える時期は、短い満期が長い満期より割高。

In [ ]:
SHOW = ["2Y", "7Y", "20Y"]
iv1m, iv1y = S.iv("1M"), S.iv("1Y")
fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw=dict(height_ratios=[3, 2]))
for c, t in zip(C, SHOW):
    axes[0].plot(iv1m.index, iv1m[t], color=c, label=f"1Mx{t}")
    axes[0].plot(iv1y.index, iv1y[t], color=c, ls=":", lw=1.2, label=f"1Yx{t}")
    axes[1].plot(iv1m.index, (iv1m[t] / iv1y[t]).rolling(5).mean(), color=c, label=t)
axes[0].set_ylabel("IV(bp/年)"); axes[0].legend(ncol=3)
axes[1].axhline(1, color=GRAY, lw=0.8); axes[1].set_ylabel("1M/1Y(5日平均)"); axes[1].legend(ncol=3)
axes[0].set_title("IVの推移")
plt.show()

### 2-3. IV と その後に実現したボラ
①の取引の中身そのものです。建てた日の1MのIVと、**その後21営業日に実現したボラ**(1MのCMのFwdの日次変化の標準偏差、年率)を重ねます。
実線(IV)が点線(実現)の上にある期間が長いほど、1Mの売りが儲かります。

In [ ]:
T = "7Y"
rv_fwd = S.fwd("1M")[T].diff().rolling(21).std().shift(-21) * np.sqrt(252)   # 翌日〜21日後の実現
fig, ax = plt.subplots()
ax.plot(iv1m.index, iv1m[T], color=C[0], label="1MのIV(建てた日)")
ax.plot(rv_fwd.index, rv_fwd, color=C[1], ls="--", label="その後21日の実現ボラ")
ax.fill_between(iv1m.index, iv1m[T], rv_fwd, where=iv1m[T] > rv_fwd, color=POS, alpha=0.12, lw=0)
ax.fill_between(iv1m.index, iv1m[T], rv_fwd, where=iv1m[T] <= rv_fwd, color=NEG, alpha=0.12, lw=0)
ax.set_ylabel("bp/年"); ax.legend(); ax.set_title(f"1Mx{T}: IV vs その後の実現ボラ(青=売りが勝ち、赤=負け)")
plt.show()
d = (iv1m[T] - rv_fwd).dropna()
print(f"IV − 実現 の平均 {d.mean():+.1f}bp、IVが上だった日の割合 {(d > 0).mean():.0%}(粗い目安。下の時価評価が正)")

## 3. 時価評価のしくみ(全戦略の共通部品)

`Surface.run(満期, 原資産, h=保有日数, k=デルタの取り直し間隔, theta=休日の重み)` が、**全ての建て日について**
ATMストラドル(ロング、年金1単位)を同じ契約のまま日々評価します。

1. 建て日 t0: ストライク K = その日の契約のFwd、σ = その日のIV。
2. 翌日: 契約のFwd(満期日が近づく分、CMのFwdを営業日数で補間)と、契約のIV(総分散を市場の時間で補間)を取り直す。
3. 1日の増分 = **ガンマ・セータ**(IVを据え置いて、Fwdの動きと時間の経過で変わった分 − デルタヘッジ)
   \+ **IVの変化**(新しいIVで評価し直した分)。
4. 合計を建てた時のベガ(ボラ1bpあたり)で割る → 「ボラのbp」。満期保有なら ≒ 実現ボラ − IV。

下で1本の取引を選んで、日々の動きを見てください。

In [ ]:
E, T, H, SIDE = "1M", "7Y", 999, -1        # 満期, 原資産, 保有日数(999=満期まで), +1=買い / −1=売り
T0 = S.D[-60]                               # 建て日(例: pd.Timestamp("2024-07-24"))
T0 = S.D[S.D.searchsorted(pd.Timestamp(T0))]
path = S.trade_path(E, T, T0, h=H, sign=SIDE)
R, _ = S.run(E, T, H)
K0 = S.A[T]["F"][S.D.get_loc(T0), GRID.index(E)]
print(f"{E}x{T} {'買い' if SIDE > 0 else '売り'}  建て日 {T0.date()}  IV {S.iv(E).loc[T0, T]:.1f}bp  ストライク {K0:.1f}bp  "
      f"保有 {len(path)}日  合計 {path['累積P&L'].iloc[-1]:+.2f} ボラbp" + ("" if T0 in R.index else "(満期・保有日数に届いていない建て)"))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(path.index, path["Fwd(bp)"], color=C[0]); axes[0].set_title("契約のFwd(bp)")
axes[1].plot(path.index, path["ガンマ・セータ"].cumsum(), color=C[0], label="ガンマ・セータ(累積)")
axes[1].plot(path.index, path["IVの変化"].cumsum(), color=C[1], label="IVの変化(累積)")
axes[1].plot(path.index, path["累積P&L"], color="#0b0b0b", lw=2, label="合計")
axes[1].axhline(0, color=GRAY, lw=0.8); axes[1].legend(); axes[1].set_title("P&L / ベガ(ボラbp)")
for a in axes: a.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
plt.show()
path.tail(5)

## ① 1Mのストラドル売り(満期保有・毎日デルタヘッジ)

**取引**: 毎日、1Mx原資産のATMストラドルを売り、満期まで毎日デルタヘッジする。儲け ≒ IV − 実現ボラ。

### ①-1 年ごと・原資産ごとの P&L/ベガ

In [ ]:
yr = {}
tv = {}
for t in TS:
    R, _ = S.run("1M", t)
    y = -R.x - COST_BP                  # 満期保有なので売買は建ての1回だけ
    yr[t] = y.groupby(y.index.year).mean()
    tv[t] = stat(y)["t"]
tab = pd.DataFrame(yr).T
tab["全期間のt値"] = pd.Series(tv)
fig, ax = plt.subplots(figsize=(8, 3.4))
v = tab.drop(columns="全期間のt値")
im = ax.imshow(v.values, cmap=DIV, norm=TwoSlopeNorm(0, -np.nanmax(abs(v.values)), np.nanmax(abs(v.values))), aspect="auto")
ax.set_xticks(range(v.shape[1]), v.columns); ax.set_yticks(range(v.shape[0]), [f"1Mx{t}" for t in v.index]); ax.grid(False)
for i in range(v.shape[0]):
    for j in range(v.shape[1]):
        ax.text(j, i, f"{v.values[i, j]:+.1f}", ha="center", va="center", fontsize=9)
ax.set_title("1M売りの P&L/ベガ(年平均、ボラbp)")
plt.show()
tab

### ①-2 ヘッジの頻度と、金利の方向
- ヘッジを5日ごと・無し(満期の払い出しだけ)にしても割高さが残るか → 残るなら「日次の往って来い(平均回帰)」だけの話ではない。
- 保有中に金利が上がった月・下がった月で分ける → 「金利上昇局面だから儲かった」のではないかを見る。

In [ ]:
rows = {}
for t in TS:
    r = {}
    for k, lab in [(1, "毎日"), (5, "5日ごと"), (999, "ヘッジ無し")]:
        R, _ = S.run("1M", t, 999, k)
        r[lab] = -R.x.mean() - COST_BP
    R, M = S.run("1M", t)
    i0 = S.D.get_indexer(R.index)
    last = np.array([row[~np.isnan(row)][-1] for row in M["F"][i0]])
    dF = pd.Series(last - R.K.values, R.index)            # 保有中の契約のFwdの変化(bp)
    q = pd.qcut(dF, 3, labels=["下げ", "横ばい", "上げ"])
    g = (-R.x).groupby(q, observed=True).mean()
    r.update({f"金利{k}": v for k, v in g.items()})
    rows[t] = r
pd.DataFrame(rows).T

### ①-3 カレンダー(1M売り + 1Y買い)と、毎日1/21ずつ建てる運用の累積P&L
1Yの買い(21日保有、ベガ同量)を重ねると、IVの水準が上がる局面(2022・2025)の損を1Yの脚が埋めます。
累積P&Lは「1Mのベガ ¥1百万/ボラbp を毎月建てる量」を21日に分けて毎日建てた場合(単位: 百万円)。

In [ ]:
T = "7Y"
R1, _ = S.run("1M", T)
R1y, _ = S.run("1Y", T, 21)
short = S.ladder({("1M", T, 999, 1, 0.0): pd.Series(-1.0, R1.index)})
cal = S.ladder({("1M", T, 999, 1, 0.0): pd.Series(-1.0, R1.index), ("1Y", T, 21, 1, 0.0): pd.Series(1.0, R1y.index)})
lo, hi = S.D[21], S.D[-22]
short, cal = short[lo:hi], cal[lo:hi]
fig, ax = plt.subplots()
ax.plot(short.cumsum(), color=C[0], label=f"1Mx{T}売り")
ax.plot(cal.cumsum(), color=C[1], label=f"カレンダー(1Mx{T}売り / 1Yx{T}買い)")
ax.axhline(0, color=GRAY, lw=0.8); ax.legend(); ax.set_ylabel("百万円"); ax.set_title("毎日1/21ずつ建てた時の累積P&L(コスト控除前)")
plt.show()
pd.DataFrame({"1M売り": ladder_summary(short), "カレンダー": ladder_summary(cal)}).T

## ② タイミング: 1M/1YのIV比

**取引**: ①と同じ1M売りを、1M/1YのIV比が高い日に厚く・低い日に薄く建てる。
信号は建てた日に分かる値だけ(1MのIV ÷ 1YのIV)。下は五分位ごとの①のP&Lと、時間・IV水準・金利を入れた回帰。

In [ ]:
def z(x): return (x - x.mean()) / x.std()
quint, reg, yrc = {}, {}, {}
for t in TS:
    R, _ = S.run("1M", t)
    df = pd.DataFrame(dict(
        y=-R.x, sig=iv1m[t] / iv1y[t], lvl=iv1y[t], rate=S.fwd("1M")[t],
        dr20=S.fwd("1M")[t].diff(20), tt=pd.Series(np.arange(S.nD) / 252, S.D))).dropna()
    quint[t] = df.y.groupby(pd.qcut(df.sig, 5, labels=["低", "2", "3", "4", "高"]), observed=True).mean()
    b1, t1 = ols_nw(z(df.sig).values[:, None], df.y.values)
    b2, t2 = ols_nw(np.c_[z(df.sig), z(df.tt), z(df.lvl), z(df.rate), z(df.dr20)], df.y.values)
    g = df.groupby(df.index.year)
    b3, t3 = ols_nw(((df.sig - g.sig.transform("mean")) / df.sig.std()).values[:, None], (df.y - g.y.transform("mean")).values)
    reg[t] = {"単独": f"{b1[1]:+.2f} (t{t1[1]:+.1f})", "+時間・IV水準・金利・20日の金利変化": f"{b2[1]:+.2f} (t{t2[1]:+.1f})",
              "年の中だけ": f"{b3[1]:+.2f} (t{t3[1]:+.1f})", "金利水準の係数": f"{b2[4]:+.2f} (t{t2[4]:+.1f})"}
    yrc[t] = {y: np.corrcoef(gg.sig, gg.y)[0, 1] for y, gg in g}
Q = pd.DataFrame(quint).T
fig, ax = plt.subplots(figsize=(10, 3.5))
w = 0.15
for i, col in enumerate(Q.columns):
    ax.bar(np.arange(len(Q)) + (i - 2) * w, Q[col], width=w, color=SEQ(0.25 + 0.17 * i), label=f"比 {col}")
ax.set_xticks(range(len(Q)), [f"1Mx{t}" for t in Q.index]); ax.axhline(0, color=GRAY, lw=0.8)
ax.legend(ncol=5); ax.set_ylabel("ボラbp"); ax.set_title("1M/1Y比の五分位ごとの、1M売りの P&L/ベガ")
plt.show()
display(pd.DataFrame(reg).T)
print("年ごとの相関(比 と 1M売りのP&L):")
pd.DataFrame(yrc).T

## ③ PC2売り(カーブの曲がりのボラを売る)

**考え方**: 1MのCMのFwdの日次変化で過去120日のPCAを取り、各原資産の1MのIV²を因子の分散に分解する
(IV²_t ≈ Σ_k インプライドの分散_k × 負荷_{k,t}²)。因子ごとの割高さ = √(インプライド / 実現)。
PC2(曲がり)がPC1(水準)より割高な時に、PC2のガンマだけを売りたい。

**取引(真ん中の1Mx7Yのベガ=1)**
- 1Mx7Y **買い** 1
- 1Mx2Y・1Mx20Y **売り**: 量は PC1のガンマ(∝ ベガ×PC1の負荷²/IV)が0になるように
- 1Yx7Y: ネットのベガを0にする量(IVの水準の変化を消す)、21日保有。両端の売りのベガが真ん中の買いより大きいので、ふつうは**買い**

PC2が動くと真ん中と両端が逆に動くので、両端の売りのガンマ(PC2の負荷が大きい)が効く=PC2の実現が小さいほど儲かる。

In [ ]:
pc = rolling_pca(S, 120)
rich = pca_richness(pc)
day = max(pc)
u1, u2, u3 = pc[day]["u"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for c, (lab, u) in zip(C, [("PC1", u1), ("PC2", u2), ("PC3", u3)]):
    axes[0].plot(range(len(u)), u.values, marker="o", ms=5, color=c, label=lab)
axes[0].set_xticks(range(len(u1)), u1.index); axes[0].axhline(0, color=GRAY, lw=0.8); axes[0].legend()
axes[0].set_title(f"PCAの負荷(過去120日、{day.date()})")
axes[1].plot(rich.PC1.rolling(5).mean(), color=C[0], label="PC1の割高さ")
axes[1].plot(rich.PC2.rolling(5).mean(), color=C[1], label="PC2の割高さ")
axes[1].plot(rich.ratio.rolling(5).mean(), color="#0b0b0b", lw=1, label="比 PC2/PC1")
axes[1].axhline(1, color=GRAY, lw=0.8); axes[1].legend(); axes[1].set_title("√(インプライド/実現)(5日平均)")
plt.show()
print("中央値: PC1", round(rich.PC1.median(), 2), " PC2", round(rich.PC2.median(), 2), " 比>1の割合", round((rich.ratio > 1).mean(), 2))

### ③-1 建て方の例(想定元本)
真ん中の1Mx7Yのベガを ¥1百万/ボラbp とした時の各脚。`EX_DAY` を変えて量の変化を見てください。

In [ ]:
EX_DAY = [d for d in pc if d <= S.D[-22]][-1]
VEGA_YEN = 1e6
rows = []
for (e, h, t), q in pc2_legs(pc[EX_DAY], iv1m.loc[EX_DAY]):
    r = quotes[(quotes.d == EX_DAY) & (quotes.label == f"{e}x{t}")].iloc[0]
    vega_per_notional = r.A * 2 * PHI0 * np.sqrt(r.N / 252) * 1e-4      # 想定元本1円・ボラ1bpあたりのベガ(円)
    rows.append({"脚": f"{e}x{t}", "売買": "買い" if q > 0 else "売り", "ベガ(千円/bp)": q * VEGA_YEN / 1e3,
                 "想定元本(億円)": abs(q) * VEGA_YEN / vega_per_notional / 1e8, "IV": r.vol, "保有": "満期" if h == 999 else f"{h}日"})
print(f"建て日 {EX_DAY.date()}  PC2/PC1の割高さの比 {rich.ratio[EX_DAY]:.2f}")
pd.DataFrame(rows).set_index("脚")

### ③-2 結果: 期間ごと・変形・タイミング
「比>1」の閾値は2025-04以降で決めたので、**2022-07〜2023が本当の検証外**。2024は2024-07〜08に前の年限が大きく動き、PC1の中立が崩れて負けた。

In [ ]:
PER = [("2022-07〜2023", "2022-07-01", "2023-12-31"), ("2024", "2024-01-01", "2024-12-31"),
       ("2025-01〜04-20", "2025-01-01", "2025-04-20"), ("2025-04-21〜", "2025-04-21", "2100-01-01"), ("全期間", "2000", "2100")]
B, Lbase, Wb = pc2_book(S, pc)
turn = sum(abs(w).mean() for w in Wb.values()) + abs(Wb[("1Y", "7Y", 21, 1, 0.0)]).mean()   # 4脚の建て+1Yの手仕舞い
net = B.x - COST_BP * turn
print(f"1建ての売買量 ベガ{turn:.2f}単位 → 損益分岐のコスト 片道 {B.x.mean() / turn:.2f} ボラbp")
display(by_period(net, PER)[["n", "mean", "t", "hit"]])
print("分解(全期間平均): ガンマ・セータ", round(B.gth.mean(), 2), " IVの変化", round(B.vg.mean(), 2))

variants = {
    "基本 7Y / 2Y+20Y +1Yx7Y": {}, "ヘッジ無し(1Mだけ)": dict(hedge=None), "ヘッジ 3Mx7Y": dict(hedge=("3M", 21)),
    "真ん中5Y": dict(belly="5Y"), "真ん中10Y": dict(belly="10Y"), "両端 1Y+20Y": dict(wings=("1Y", "20Y")),
    "両端 2Y+15Y": dict(wings=("2Y", "15Y")), "ベガ等量(PC1中立でない)": dict(mode="vega"), "仮説の向き(真ん中売り)": dict(long_belly=False)}
vt = {k: by_period(pc2_book(S, pc, **kw)[0].x, PER)["mean"] for k, kw in variants.items()}
display(pd.DataFrame(vt).T.rename(columns=lambda c: c + " 平均"))

on = lambda d: rich.ratio.get(d, 0) > 1
Bon, Lon, _ = pc2_book(S, pc, entry_ok=on)
print("比>1の日だけ建てる:")
display(by_period(Bon.x, PER)[["n", "mean", "t", "hit"]])
lo, hi = pd.Timestamp("2022-07-11"), S.D[-22]
fig, ax = plt.subplots()
ax.plot(Lbase[lo:hi].cumsum(), color=C[0], label="常に建てる")
ax.plot(Lon[lo:hi].cumsum(), color=C[1], label="比>1の日だけ")
ax.axvspan(lo, pd.Timestamp("2023-12-31"), color=GRAY, alpha=0.08, lw=0)
ax.text(lo, ax.get_ylim()[1], " 検証外", va="top", color=GRAY)
ax.axhline(0, color=GRAY, lw=0.8); ax.legend(); ax.set_ylabel("百万円"); ax.set_title("PC2売り: 毎日1/21ずつ建てた時の累積P&L(真ん中のベガ¥1百万/bp/月)")
plt.show()
pd.DataFrame({"常に": ladder_summary(Lbase[lo:hi]), "比>1": ladder_summary(Lon[lo:hi])}).T

### ③-3 ①との関係
PC2売りと1M売りの日次P&Lの相関が ≈0 なら、両方持つと分散が効く。

In [ ]:
pd.DataFrame({"PC2売り": Lbase, "1Mx7Y売り": S.ladder({("1M", "7Y", 999, 1, 0.0): pd.Series(-1.0, S.run("1M", "7Y")[0].index)}),
              "1Mx1Y売り": S.ladder({("1M", "1Y", 999, 1, 0.0): pd.Series(-1.0, S.run("1M", "1Y")[0].index)})})[lo:hi].corr()

## ④ 日銀会合の前後

**取引**: 会合の `BEFORE` 日前の引けに1Mのストラドルを売り(デルタヘッジ)、`AFTER` 日後の引けに買い戻す。
比べる相手は、会合を含まない同じ長さの窓の売り。会合前にIVが上がり、会合後に下がる(ベガの儲け)のを取る。

### ④-1 会合の前後の1MのIVの平均の動き(会合の10日前=0)

In [ ]:
MI = [S.D.get_loc(m) for m in BOJ_MEETINGS if m in S.D]
ks = range(-10, 6)
fig, ax = plt.subplots(figsize=(9, 3.8))
for c, t in zip(C, TS):
    pth = np.array([[iv1m[t].iloc[i + k] - iv1m[t].iloc[i - 10] for k in ks] for i in MI if i + 5 < S.nD and i >= 10])
    ax.plot(list(ks), pth.mean(0), marker="o", ms=4, color=c, label=t)
ax.axvline(0, color=GRAY, lw=0.8); ax.axhline(0, color=GRAY, lw=0.8)
ax.set_xlabel("会合日からの営業日"); ax.set_ylabel("ボラbp"); ax.legend(ncol=6); ax.set_title(f"1MのIVの平均の変化(会合{len(MI)}回)")
plt.show()

In [ ]:
BEFORE, AFTER = 3, 3
rows = {}
for t in TS:
    X, Cn = meeting_trades(S, t, BEFORE, AFTER)
    y = X.P_L - 2 * COST_BP
    rows[t] = dict(平均=y.mean(), t値=y.mean() / y.std() * np.sqrt(len(y)), 勝率=(y > 0).mean(), 最悪=y.min(),
                   通常の窓=Cn.mean(), うちガンマ=X.ガンマ_セータ.mean(), うちIV=X.IVの変化.mean(),
                   YCC期_2022_23=y[X.index.year <= 2023].mean(), 利上げ期_2024_=y[X.index.year >= 2024].mean())
display(pd.DataFrame(rows).T)
T = "20Y"
X, _ = meeting_trades(S, T, BEFORE, AFTER)
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(range(len(X)), X.P_L, color=[POS if v >= 0 else NEG for v in X.P_L], width=0.7)
ax.set_xticks(range(len(X)), [d.strftime("%y-%m") for d in X.index], rotation=90, fontsize=8)
ax.axhline(0, color=GRAY, lw=0.8); ax.set_ylabel("ボラbp"); ax.grid(axis="x", visible=False)
ax.set_title(f"1Mx{T}: 会合ごとの売りのP&L({BEFORE}日前に売り→{AFTER}日後に買い戻し)")
plt.show()

## ⑤ 週末と、IVの形

### ⑤-1 夜の種類ごとの1晩のP&L
市場は休日を営業日の何日分で値付けしているか。休日の重み θ(0=営業日だけ、1=暦日)で評価し直すと、
θ≈0.3 で金→月の夜の値付け(≈1.6日分)が実現(≈1.0〜1.1日分)より多い。下は1M売り(満期保有)の各夜の平均P&L。

In [ ]:
THETA = 0.3
typ = night_types(S)
rows = {}
for t in TS:
    R, M = S.run("1M", t, theta=THETA)
    i0, st = np.nonzero(~np.isnan(M["inc"]))
    j = i0 + st + 1
    v = pd.Series(-M["inc"][i0, st])
    rows[t] = v.groupby(typ.values[j]).mean()
N = pd.DataFrame(rows).T
fig, axes = plt.subplots(1, len(N.columns), figsize=(13, 3), sharey=True, sharex=True)
for ax, col in zip(axes, N.columns):
    bars(ax, N[col], col, "{:+.2f}")
plt.suptitle(f"1M売りの1晩のP&L/ベガ(θ={THETA})", y=1.03)
plt.show()

### ⑤-2 金曜に売って月曜に買い戻す
1晩だけの売り(デルタヘッジ)を、週末をまたぐ夜と平日の夜で比べる。

In [ ]:
rows = {}
nxt = typ.shift(-1)
for t in TS:
    R, _ = S.run("1M", t, 1, theta=THETA)
    y = -R.x - 2 * COST_BP
    n = nxt.reindex(y.index)
    fri, wk = y[n == "週末明け(金→月)"], y[n == "平日の夜"]
    rows[t] = dict(金曜売り=fri.mean(), 金曜_t=fri.mean() / fri.std() * np.sqrt(len(fri)), 金曜_勝率=(fri > 0).mean(),
                   平日売り=wk.mean(), 差=fri.mean() - wk.mean(), 金曜_うちガンマ=-R.gth[fri.index].mean(), 金曜_うちIV=-R.vg[fri.index].mean())
pd.DataFrame(rows).T

### ⑤-3 IVの年限方向の形の平均回帰
1YのIVを log(年限) の2次式で当てはめ、残差がマイナス(割安)の原資産を買い・プラス(割高)を売る(ベガ中立、買い1・売り1)、h日保有。
全期間で効くが小さく、2026年は≈0。一部はクオートのノイズの可能性がある。

In [ ]:
EXP, HOLD = "1Y", 21
T9 = [t for t in TENORS if t in S.tenors]
I = S.iv(EXP)[T9]
lx = np.log([int(t[:-1]) for t in T9]); Xm = np.c_[np.ones(len(T9)), lx, lx ** 2]
ok = I.notna().all(1)
res = pd.DataFrame(np.nan, I.index, T9)
res[ok] = I[ok].values - (Xm @ np.linalg.lstsq(Xm, I[ok].values.T, rcond=None)[0]).T
w = -res.sub(res.mean(1), axis=0)
w = w.div(w.abs().sum(1), axis=0) * 2
P = pd.DataFrame({t: S.run(EXP, t, HOLD)[0].x for t in T9}).reindex(S.D)
pnl = (w * P).sum(1, min_count=len(T9)).dropna() - 2 * 2 * COST_BP     # 買い1・売り1を建てて手仕舞う
st = stat(pnl, HOLD)
print(f"{EXP} h={HOLD}: 平均 {st['mean']:+.2f} ボラbp (t{st['t']:+.1f}, 勝率 {st['hit']:.0%})")
display(pnl.groupby(pnl.index.year).mean().to_frame("年平均").T)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
day = I[ok].index[-1]
fit = Xm @ np.linalg.lstsq(Xm, I.loc[day].values, rcond=None)[0]
axes[0].plot(range(len(T9)), I.loc[day].values, "o", ms=7, color=C[0], label="IV")
axes[0].plot(range(len(T9)), fit, color=C[1], label="2次式")
axes[0].set_xticks(range(len(T9)), T9); axes[0].legend(); axes[0].set_title(f"{EXP}のIVの断面 {day.date()}")
axes[1].plot(pnl.rolling(63).mean(), color=C[0]); axes[1].axhline(0, color=GRAY, lw=0.8)
axes[1].set_title("P&L/ベガの63日移動平均")
plt.show()

## まとめと、続きをやるなら

| # | 取引 | 結果(コスト控除前) | 弱点 |
|---|---|---|---|
| ① | 1M売り | 2023〜26は毎年プラス。2022の中期だけ≈0。ヘッジの頻度・金利の方向に関係なく残る | ボラが上がる年(2022・2025)。カレンダーで和らぐ |
| ② | 1M/1Y比のタイミング | 時間・水準・金利を入れても、年の中だけで見ても残る | 金利水準が高いと①自体が小さい |
| ③ | PC2売り | 検証外の2022-07〜2023で +9.2(t3.7)、比>1で +11.7 | 2024-07〜08に前の年限が動いてPC1の中立が崩れた。損益分岐は片道約1.5ボラbp |
| ④ | 会合前の20Y売り | 3日前→3日後で +5.4(通常0.9、勝率76%)。儲けはIVの低下 | サプライズの会合(2022-12・2023-07・2024-07)で大きく負ける |
| ⑤ | 週末・形 | 金曜売り +0.4〜0.5(t1〜2)。形の平均回帰は小さい | 小さい。ノイズの可能性 |

- 売買コスト(`COST_BP`)を入れて、どこまで残るかを最初に確かめる。
- 閾値・年限の選び方は後付けなので、新しいデータ(2026-10以降)で確かめ直す。